<div align="center"><span style="font-family: Arial; color:#0000FF"><b>
    <span style="font-size: x-large">Metodos Numericos II</span>
    <br>
    <span style="font-size: large">Segundo de Grado en Matematicas - Curso 2023/24</span>
    <br>
    <span style="font-size: medium">Facultad de Ciencias de la Universidad de Malaga</span>
    <br>
    <span style="font-size: small">Dpto. de Analisis Matematico, Estadistica e Investigacion Operativa, y Matematica Aplicada</span>
    <br>
    <span style="font-size: medium; color:#000000">Profs. Manuel J. Castro y Francisco J. Palma (Area Conocimiento de Matematica Aplicada)</span>
    <br>
    <span style="font-size: medium; color:#FF0000">Seccion numero 8</span>
    </b></span></div>

In [2]:
from algoritmos import *

<div align="left"><span style="font-family: Arial; color:#000000; font-size: medium">
    El objetivo de esta seccion es desarrollar funciones <span style="font-family: Courier">Python</span> para resolver sistemas de ecuaciones lineales mediante los <b>metodos iterativos clasicos de Jacobi, de Gauss-Seidel y de relajacion</b>.
    </span></div>

<div align="left"><span style="font-family: Arial; color:#000000; font-size: medium">
    Recordamos que, en general, los metodos iterativos para resolver un sistema de ecuaciones lineales compatible y determinado de $n$ ecuaciones con $n$ incognitas $A\,X=B$ (donde los datos del problema son $A\in\mathcal{M}_n(\mathbb{K})$ inversible y $B\in\mathbb{K}^n$, y la incognita es $X\in\mathbb{K}^n$), se basan en construir una matriz $C\in\mathcal{M}_n(\mathbb{K})$ y un vector $V\in\mathbb{K}^n$ tales que
$$
A\,X = B \quad \Leftrightarrow \quad X = C\,X +V\,.
$$
    <br>
    A partir de esta formulacion equivalente del problema, se construye una sucesion $\{X_k\}_{k\in\mathbb{N}}$ de aproximaciones de la solucion de la forma siguiente:
$$
\left\{ \begin{array}{l} X_0 \in \mathbb{K}^n \quad \mbox{dado}\,, \\ X_{k+1} = C\,X_k + V\,, \quad k=0,1,2,\ldots. \end{array} \right.
$$
    <br>
    Es claro que esta construccion de la sucesion permite asegurar que en caso de convergencia lo hara hacia la unica solucion del problema dado, y sabemos que esa convergencia se produce si y solo si $\rho(C)<1$.
    <br>
    Para la construccion de la matriz $C$ y del vector $V$, se parte de una descomposicion de la matriz $A=M-N$, donde $M,N\in\mathcal{M}_n(\mathbb{K})$, con $M$ inversible: se escribe entonces
\[
A\,X = B \quad \Leftrightarrow \quad X = M^{-1}\,N\,X + M^{-1}\,B\,,
\]
con lo que $C=M^{-1}\,N$ y $V=M^{-1}\,B$.
    <br>
    Los metodos iterativos clasicos utilizan la descomposicion de la matriz $A$ en la forma
\[
A = D - E - F\,,
\]
donde las matrices $D,E,F\in\mathcal{M}_n(\mathbb{K})$ son, respectivamente, diagonal, estrictamente triangular inferior y estrictamente triangular superior. Hacemos siempre la hipotesis que los elementos diagonales de $A$ son no nulos, lo que asegura que la matriz $D$ siempre es inversible.
    </span></div>

<div align="left"><span style="font-family: Arial; color:#000000; font-size: medium">
    El <b>metodo iterativo de Jacobi</b> se basa en la eleccion $M=D$ y $N=E+F$, con lo que se tiene que $C=D^{-1}\,(E+F)$ (esta matriz se suele notar mediante $J$) y $V=D^{-1}\,B$. La sucesion generada $\{X_k\}_{k\in\mathbb{N}}$ viene dada por
\[
\left\{ \begin{array}{l} X_0 \in \mathbb{K}^n \quad \mbox{dado}\,, \\ X_{k+1} = D^{-1}\,(E+F)\,X_k + D^{-1}\,B\,, \quad k=0,1,2,\ldots, \end{array} \right.
\]
o equivalentemente
\[
\left\{ \begin{array}{l} X_0 \in \mathbb{K}^n \quad \mbox{dado}\,, \\ D\,X_{k+1} = B + (E+F)\,X_k\,, \quad k=0,1,2,\ldots. \end{array} \right.
\]
    <br>
    Si ponemos $X_k=(x_i^{(k)})_{k=1}^n$, entonces
\[
x_i^{(k+1)} = \frac{1}{a_{i,i}}\,\left( b_i - \sum_{j=1\,,\,j\ne i}^n a_{i,j}\,x_j^{(k)} \right)\,.
\]
    </span></div>

<div align="left"><span style="font-family: Arial; color:#000000; font-size: medium">
    <span style="color:#FF0000"><b>Caso 1.</b></span> Elaborar un programa de nombre <span style="font-family: Courier">jacobi()</span> que implemente el algoritmo del <b>metodo iterativo de Jacobi</b>.
    </span></div>

<div align="left"><span style="font-family: Arial; color:#000000; font-size: medium">
    <span style="color:#FF0000"><b>Observacion.</b></span> En todos los programas que siguen, los mismos llevan como parametros de entrada la matriz $A$, el segundo miembro $B$, el vector $X_0$ con el que iniciar las iteraciones, el numero maximo de iteraciones a realizar y la tolerancia del test de parada (que es la norma infinito de la diferencia entre dos iteraciones sucesivas); en el caso del metodo de relajacion, tambien hay que dar el parametro de relajacion $\omega$.
    </span></div>

In [4]:
def jacobi(A, B, XOLD, itermax, tol):
    m, n = shape(A)
    p, q = shape(B)
    r, s = shape(XOLD)

    if m != n or n != p or q != 1 or n != r or s != 1 or min(abs(diag(A))) < 1e-200:
        return False, 'ERROR jacobi: no se resuelve el sistema.'

    if A.dtype == complex or B.dtype == complex or XOLD.dtype == complex:
        tipo = 'complex'
    else:
        tipo = 'float'

    k = 0
    error = tol

    while k < itermax and error >= tol:
        k = k+1
        XNEW = array(B, dtype=tipo)

        for i in range(n):
            if i != 0:
                XNEW[i, 0] -= A[i, :i]@XOLD[:i, 0]

            if i != n-1:
                XNEW[i, 0] -= A[i, i+1:]@XOLD[i+1:, 0]

            XNEW[i, 0] = XNEW[i, 0]/A[i, i]

        error = norma_vec(XNEW - XOLD, inf)
        XOLD = array(XNEW)

    print('Iteracion: k = ', k)
    print('Error absoluto: error = ', error)

    if k == itermax and error >= tol:
        return False, 'ERROR jacobi: no se alcanza convergencia.'
    else:
        print('Convergencia numerica alcanzada p jacobi.')
        return True, XNEW

<div align="left"><span style="font-family: Arial; color:#000000; font-size: medium">
    <span style="color:#FF0000"><b>Caso 2.</b></span> Resolver mediante el metodo iterativo de Jacobi un sistema lineal $A\,X=B$, cuya matriz de coeficientes $A$ (del tamano que se quiera) es tridiagonal, con elementos diagonales iguales a 2 y elementos sub-diagonales y supra-diagonales iguales a $-1$ (esta matriz es definida positiva); sabemos que hay convergencia en este caso. Tomamos como segundo miembro $B$ un vector cuyas componentes son las sumas de las respectivas filas de la matriz $A$, lo que nos asegura que la solucion exacta del sistema $X$ es el vector con todas las componentes iguales a 1. Tomamos como vector inicial $X_0$ el vector nulo, establecemos un numero maximo de iteraciones de 1000 y un valor para la constante de tolerancia de $10^{-5}$.
    </span></div>

In [ ]:
n = 50
A = 2*eye(n) - eye(n, k=-1) - eye(n, k=1)
print('Matriz: A = \n', A)
B = reshape(sum(A, axis=1), (n,1))
print('Segundo miembro: B = \n', B)
X_0 = zeros((n, 1))
print('Vector inicial: X_0 = \n', X_0)
exito, X = jacobi(A, B, X_0, 10000, 1e-5)
if exito:
    print('Solucion aproximada: X_k = \n', X)
    print('Comprobacion: ||B-A X||_2 = ', norm(B-A@X, 2))
else:
    print(X)

<div align="left"><span style="font-family: Arial; color:#000000; font-size: medium">
  El <b>metodo iterativo de Gauss-Seidel</b> se basa en la eleccion $M=D-E$ y $N=F$, con lo que se tiene que
  $C=(D-E)^{-1}\,F$ (esta matriz se suele notar mediante $\mathcal{L}_1$) y $V=(D-E)^{-1}\,B$. La sucesion generada $\{X_k\}_{k\in\mathbb{N}}$ viene dada por
$$
\left\{ \begin{array}{l} X_0 \in \mathbb{K}^n \quad \mbox{dado}\,, \\ X_{k+1} = (D-E)^{-1}\,F\,X_k + (D-E)^{-1}\,B\,, k=0,1,2,\ldots, \end{array} \right.
$$
o equivalentemente
$$
\left\{ \begin{array}{l} X_0 \in \mathbb{K}^n \quad \mbox{dado}\,, \\ D\,X_{k+1} = B + E\,X_{k+1} + F\,X_k\,, \quad k=0,1,2,\ldots. \end{array} \right.
$$
  <br>
  Si ponemos $X_k=(x_i^{(k)})_{k=1}^n$, entonces
$$
x_i^{(k+1)} = \frac{1}{a_{i,i}}\,\left( b_i - \sum_{j=1}^{i-1} a_{i,j}\,x_j^{(k+1)} - \sum_{j=i+1}^n a_{i,j}\,x_j^{(k)} \right)\,.
$$
    </span></div>

<div align="left"><span style="font-family: Arial; color:#000000; font-size: medium">
    <span style="color:#FF0000"><b>Caso 3.</b></span> Elaborar un programa de nombre <span style="font-family: Courier">gauss_seidel()</span> que implemente el algoritmo del <b>metodo iterativo de Gauss-Seidel</b>.
    </span></div>

In [16]:
def gauss_seidel(A, B, XOLD, itermax, tol):
  m, n = shape(A)
  p, q = shape(B)
  r, s = shape(XOLD)

  if m != n or n != p or q != 1 or n != r or s != 1 or min(abs(diag(A))) <= 10e-200:
    return False, 'ERROR gauss_seidel: no se resuelve el sistema.'

  k = 0
  error = 1.

  while k < itermax and error >= tol:
    k = k+1
    XNEW = array(B)

    for i in range(n):
      if i != 0:
        XNEW[i, 0] -= A[i, :i]@XNEW[:i, 0]
      if i != n-1:
        XNEW[i, 0] -= A[i, i+1:]@XOLD[i+1:, 0]
      XNEW[i, 0] = XNEW[i, 0]/A[i, i]

    error = norma_vec(XNEW - XOLD, inf)
    XOLD = array(XNEW)

  print('Iteracion: k = ', k)
  print('Error absoluto: error = ', error)

  if k == itermax and error >= tol:
    return False, 'ERROR gauss_seidel: no se alcanza convergencia.'
  else:
    print('Convergencia numerica alcanzada: gauss_seidel.')
    return True, XNEW

<div align="left"><span style="font-family: Arial; color:#000000; font-size: medium">
    <span style="color:#FF0000"><b>Caso 4.</b></span> Resolver el mismo sistema que se plantea en el caso 2 mediante el metodo de Gauss-Seidel</b>.
    </span></div>

In [ ]:
n = 500
A = 2*eye(n)-eye(n,k=-1)-eye(n,k=1)
B = reshape(sum(A,axis=1),(n,1))
XOLD = zeros((n,1))
exito, X = gauss_seidel(A,B,XOLD,1000,1e-5)
print(X)

<div align="left"><span style="font-family: Arial; color:#000000; font-size: medium">
  Finalmente la familia de <b>metodos iterativos de relajacion</b> se basa en la eleccion $M=\frac{1}{\omega}\,D-E$ y $N=\frac{1-\omega}{\omega}\,D+F$, donde $\omega\in\mathbb{R}-\{0\}$, con lo que se tiene que $C=\left(\frac{1}{\omega}\,D-E\right)^{-1}\,\left(\frac{1-\omega}{\omega}\,D+F\right)$ (esta matriz se suele notar mediante $\mathcal{L}_\omega$) y $V=\left(\frac{1}{\omega}\,D-E\right)^{-1}\,B$. La sucesion generada $\{X_k\}_{k\in\mathbb{N}}$ viene dada por
$$
\left\{ \begin{array}{l} X_0 \in \mathbb{K}^n \quad \mbox{dado}\,, \\ X_{k+1} = \left(\frac{1}{\omega}\,D-E\right)^{-1}\,\left(\frac{1-\omega}{\omega}\,D+F\right)\,X_k + \left(\frac{1}{\omega}\,D-E\right)^{-1}\,B\,, \quad k=0,1,2,\ldots, \end{array} \right.
$$
o equivalentemente
$$
\left\{ \begin{array}{l} X_0 \in \mathbb{K}^n \quad \mbox{dado}\,, \\ \frac{1}{\omega}\,D\,X_{k+1} = B + E\,X_{k+1} + \frac{1-\omega}{\omega}\,D\,X_k + F\,X_k\,, \quad k=0,1,2,\ldots. \end{array} \right.
$$
  <br>
  Si ponemos $X_k=(x_i^{(k)})_{k=1}^n$, entonces
$$
x_i^{(k+1)} = \frac{\omega}{a_{i,i}}\,\left( b_i - \sum_{j=1}^{i-1} a_{i,j}\,x_j^{(k+1)} + \frac{1-\omega}{\omega}\,a_{i,i}\,x_i^{(k)} - \sum_{j=i+1}^n a_{i,j}\,x_j^{(k)} \right)\,.
$$
    </span></div>

In [20]:
def relajacion(A, B, XOLD, omega, itermax, tol):
    m, n = shape(A)
    p, q = shape(B)
    r, s = shape(XOLD)

    if m != n or n != p or q != 1 or n != r or s != 1 or min(abs(diag(A))) < 1e-200:
        return False, 'ERROR relajacion: no se resuelve el sistema.'

    k = 0
    error = 1.

    while k < itermax and error >= tol:
        k = k+1
        XNEW = array(B)

        for i in range(n):
            if i != 0:
                XNEW[i, 0] -= A[i, :i]@XNEW[:i, 0]
            if i != n-1:
                XNEW[i, 0] -= A[i, i+1:]@XOLD[i+1:, 0]

            XNEW[i,0] += ((1-omega)/omega)*A[i,i]*XOLD[i,0]
            XNEW[i, 0] = omega*XNEW[i, 0]/A[i, i]

        error = norma_vec(XNEW - XOLD, inf)
        XOLD = array(XNEW)

    print('Iteracion: k = ', k)
    print('Error absoluto: error = ', error)

    if k == itermax and error >= tol:
        return False, 'ERROR relajacion: no se alcanza convergencia.'
    else:
        print('Convergencia numerica alcanzada: relajacion.')
        return True, XNEW


<div align="left"><span style="font-family: Arial; color:#000000; font-size: medium">
    <span style="color:#FF0000"><b>Caso 5.</b></span> Resolver el mismo sistema que se plantea en el caso 2 mediante el metodo de relajacion, tomando diferentes valores del parametro $\omega$</b>.
    </span></div>

In [22]:
n = 500
A = 2*eye(n)-eye(n,k=-1)-eye(n,k=1)
B = reshape(sum(A,axis=1),(n,1))
XOLD = zeros((n,1))
exito, X = relajacion(A,B,XOLD,1.369,1000,1e-5)
print(X)

Iteracion: k =  1000
Error absoluto: error =  0.0002502096414303545
ERROR relajacion: no se alcanza convergencia.


In [23]:
n = 5
A = 2*eye(n)-eye(n,k=-1)-eye(n,k=1)
B = reshape(sum(A,axis=1),(n,1))
XOLD = zeros((n,1))
exito, X = relajacion(A,B,XOLD,1.369,100000,1e-5)
print(X)

Iteracion: k =  14
Error absoluto: error =  9.88735136697727e-06
Convergencia numerica alcanzada: relajacion.
[[1.00000005]
 [1.00000203]
 [0.9999996 ]
 [0.99999923]
 [0.99999963]]


In [24]:
n = 5
A = 2*eye(n)-eye(n,k=-1)-eye(n,k=1)
B = reshape(sum(A,axis=1),(n,1))
XOLD = zeros((n,1))
exito, X = relajacion(A,B,XOLD,1,100000,1e-5)
print(X)

Iteracion: k =  38
Error absoluto: error =  8.277024824421275e-06
Convergencia numerica alcanzada: relajacion.
[[0.99998345]
 [0.99997517]
 [0.99997517]
 [0.99998138]
 [0.99999069]]


<div align="left"><span style="font-family: Arial; color:#000000; font-size: medium">
    <span style="color:#FF0000"><b>Caso 6.</b></span> Para un caso concreto de matriz tridiagonal definida positiva, calcular el parametro optimo de relajacion, y realizar diferentes ensayos de dicho metodo, utilizando el valor optimo de $\omega$, asi como valores inferiores y superiores.
    </span></div>

In [25]:
n = 5
A = 2*eye(n)-eye(n,k=-1)-eye(n,k=1)
B = reshape(sum(A,axis=1),(n,1))
XOLD = zeros((n,1))
D = 2*eye(n)
E = -tril(A,k=-1)
F = -triu(A,k=1)
J = inv(D)@(E+F)
d,P = eig(J)
ro = max(abs(d))
omega0 = 2/(1+sqrt(1-ro*ro))
print(omega0)
exito, X = relajacion(A,B,XOLD,omega0,100000,1e-5)
print(X)

1.3333333333333355
Iteracion: k =  15
Error absoluto: error =  7.140207424649603e-06
Convergencia numerica alcanzada: relajacion.
[[0.99999702]
 [0.99999671]
 [0.99999768]
 [0.99999882]
 [0.99999962]]


In [27]:
n = 500
A = 2*eye(n)-eye(n,k=-1)-eye(n,k=1)
B = reshape(sum(A,axis=1),(n,1))
XOLD = zeros((n,1))
D = 2*eye(n)
E = -tril(A,k=-1)
F = -triu(A,k=1)
J = inv(D)@(E+F)
d,P = eig(J)
ro = max(abs(d))
omega0 = 2/(1+sqrt(1-ro*ro))
print(omega0)
exito, X = relajacion(A,B,XOLD,omega0,100000,1e-5)
print(X)

1.987536945020203
Iteracion: k =  1003
Error absoluto: error =  4.094245964259002e-06
Convergencia numerica alcanzada: relajacion.
[[0.99999643]
 [0.9999929 ]
 [0.99998942]
 [0.99998598]
 [0.99998257]
 [0.99997921]
 [0.99997589]
 [0.99997262]
 [0.99996938]
 [0.99996618]
 [0.99996303]
 [0.99995991]
 [0.99995683]
 [0.9999538 ]
 [0.9999508 ]
 [0.99994784]
 [0.99994492]
 [0.99994204]
 [0.9999392 ]
 [0.9999364 ]
 [0.99993363]
 [0.99993091]
 [0.99992822]
 [0.99992556]
 [0.99992295]
 [0.99992037]
 [0.99991783]
 [0.99991532]
 [0.99991285]
 [0.99991042]
 [0.99990802]
 [0.99990566]
 [0.99990333]
 [0.99990104]
 [0.99989879]
 [0.99989656]
 [0.99989438]
 [0.99989222]
 [0.9998901 ]
 [0.99988802]
 [0.99988596]
 [0.99988394]
 [0.99988196]
 [0.99988   ]
 [0.99987808]
 [0.99987619]
 [0.99987434]
 [0.99987251]
 [0.99987072]
 [0.99986895]
 [0.99986722]
 [0.99986552]
 [0.99986385]
 [0.99986221]
 [0.9998606 ]
 [0.99985902]
 [0.99985747]
 [0.99985595]
 [0.99985446]
 [0.999853  ]
 [0.99985156]
 [0.99985016]
 

<div align="left"><span style="font-family: Arial; color:#000000; font-size: medium">
    <span style="color:#FF0000"><b>Caso 7.</b></span> En el caso de matrices tridiagonales y, en general, banda con semianchura de banda $p$, optimizar los programas anteriores de manera que se eviten las operaciones innecesarias.
    </span></div>

In [30]:
def jacobi_pdiag(A, B, XOLD, p, itermax, tol):
  m, n = shape(A)
  p, q = shape(B)
  r, s = shape(XOLD)

  if m != n or n != p or q != 1 or n != r or s != 1 or min(abs(diag(A))) < 1e-200:
    return False, 'ERROR jacobi_pdiag: no se resuelve el sistema.'

  k = 0
  error = 1.

  while k < itermax and error >= tol:
    k = k+1
    XNEW = array(B)

#    for i in range(n):
#      if i != 0:
#        XNEW[i, 0] -= A[i, max([i-p,0]):i]@XOLD[max([i-p,0]):i, 0
#      if i != n-1:
#       XNEW[i, 0] -= A[i, i+1:min([i+p+1,n])]@XOLD[i+1:min([i+p+
#      XNEW[i, 0] = XNEW[i, 0]/A[i, i]

    error = norma_vec(XNEW - XOLD, inf)
    XOLD = array(XNEW)

  print('Iteracion: k = ', k)
  print('Error absoluto: error = ', error)

  if k == itermax and error >= tol:
    return False, 'ERROR jacobi_pdiag: no se alcanza convergencia.'
  else:
    print('Convergencia numerica alcanzada: jacobi_pdiag.')
    return True, XNEW


<div align="left"><span style="font-family: Arial; color:#000000; font-size: medium">
<span style="color:#FF0000"><b>Caso 8.</b></span> Se consideran las matrices $$A = \left( \begin{array}{ccc} 1 & 2 & -2 \\ 1 & 1 & 1 \\ 2 & 2 & 1 \end{array} \right) \hspace{0.25cm} \text{y} \hspace{0.25cm} B = \left( \begin{array}{ccc} 2 & -1 & 1 \\ 2 & 2 & 2 \\ -1 & -1 & 2 \end{array} \right)$$.
<br>
Demostrar que para la primera matriz el metodo iterativo de Jacobi es convergente, pero el de Gauss-Seidel no lo es, mientras que para la segunda matriz ocurre justamente lo contrario. Escribir las matrices de ambos metodos iterativos para las dos matrices dadas.
    </span></div>

In [ ]:
n = 5
A = 2*eye(n)-eye(n,k=-1)-eye(n,k=1)
B = reshape(sum(A,axis=1),(n,1))
XOLD = zeros((n,1))
exito, X = jacobi_pdiag(A,B,XOLD,3,100000,1e-5)
print(X)